In [1]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import itertools
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.ensemble import RandomForestRegressor
import numpy as np
from plotly.subplots import make_subplots

In [2]:
# Principal
ventas = pd.read_csv('../data_raw/ventas.csv')
# Apoyo
clientes = pd.read_csv('../data_raw/clientes.csv')
metodos_de_pago = pd.read_csv('../data_raw/metodos_pago.csv')
productos = pd.read_csv('../data_raw/productos.csv')

In [3]:
# Dimensiones del dataset
ventas.shape

(3029, 7)

In [4]:
# Dimensiones de los dataset de apoyo
print(clientes.shape)
print(metodos_de_pago.shape)
print(productos.shape)

(326, 6)
(5, 3)
(38, 5)


In [5]:
# VISTA PRELIMINAR DE LOS DATOS
ventas.head()

,ID_Venta,Fecha,ID_Cliente,ID_Producto,Cantidad,Método_Pago,Estado
0,919,31/01/2024,10,25,5,1,Completa
1,947,31/01/2024,106,5,1,4,Completa
2,1317,31/1/2024,235,25,3,3,Completa
3,1607,31/1/2024,114,15,5,1,Completa
4,2038,31/1/2024,132,2,5,4,Completa


In [6]:
ventas.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3029 entries, 0 to 3028
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   ID_Venta     3029 non-null   int64 
 1   Fecha        3029 non-null   object
 2   ID_Cliente   3029 non-null   int64 
 3   ID_Producto  3029 non-null   int64 
 4   Cantidad     3029 non-null   int64 
 5   Método_Pago  3029 non-null   int64 
 6   Estado       3029 non-null   object
dtypes: int64(5), object(2)
memory usage: 165.8+ KB


In [7]:
ventas.describe()

,ID_Venta,ID_Cliente,ID_Producto,Cantidad,Método_Pago
count,3029.000000,3029.000000,3029.000000,3029.000000,3029.000000
mean,1492.663585,162.208320,19.675801,3.475404,3.359194
std,865.690540,94.276683,10.989542,1.702960,1.425749
min,1.000000,1.000000,1.000000,1.000000,1.000000
25%,729.000000,79.000000,10.000000,2.000000,2.000000
50%,1486.000000,162.000000,20.000000,3.000000,4.000000
75%,2243.000000,243.000000,29.000000,5.000000,5.000000
max,3000.000000,326.000000,38.000000,6.000000,5.000000


In [8]:
faltantes = ventas.isnull().sum()
faltantes

ID_Venta       0
Fecha          0
ID_Cliente     0
ID_Producto    0
Cantidad       0
Método_Pago    0
Estado         0
dtype: int64

In [9]:
# Valores faltantes en datasets de apoyo
faltantes_resumen = pd.DataFrame({
    'Dataset': ['Clientes', 'Métodos de Pago', 'Productos'],
    'Total_Registros': [len(clientes), len(metodos_de_pago), len(productos)],
    'Total_Valores_Faltantes': [
        clientes.isnull().sum().sum(),
        metodos_de_pago.isnull().sum().sum(), 
        productos.isnull().sum().sum()
    ]
})

faltantes_resumen

,Dataset,Total_Registros,Total_Valores_Faltantes
0,Clientes,326,0
1,Métodos de Pago,5,0
2,Productos,38,0


## Descripción General del Dataset
- Tamaño del Dataset: 3,029 registros de ventas con 7 variables
- Registros: Dataset completo sin valores faltantes
- Período: Datos de transacciones

### Variables Disponibles
- `ID_Venta` 
- `Fecha` 
- `ID_Cliente` 
- `ID_Producto` 
- `Cantidad` 
- `Método_Pago` 
- `Estado`

### Calidad de Datos
- No se detectaron valores nulos en ninguna columna
- No se identificaron problemas inmediatos de integridad de datos

### Tipos de Datos
- 5 variables numéricas (int64)
- 2 variables categóricas (object)

In [10]:
# Unicidad de llaves
print(f"ID_Venta: {ventas['ID_Venta'].nunique()} valores únicos de {len(ventas)} registros")

ID_Venta: 3000 valores únicos de 3029 registros


In [11]:
print(f"ID_Cliente: {ventas['ID_Cliente'].nunique()} clientes únicos")

ID_Cliente: 326 clientes únicos


In [12]:
# Ubicacion de los clientes
print(f"Regiones: {len(clientes['Región'].unique())}")
print(clientes['Región'].value_counts().sort_index())

Regiones: 6
Región
Buenos Aires    111
Centro           63
Cuyo             44
NEA              37
NOA               7
Patagonia        64
Name: count, dtype: int64


In [13]:
print(f"ID_Producto: {ventas['ID_Producto'].nunique()} productos únicos")

ID_Producto: 38 productos únicos


In [14]:
# Inventario de productos
productos.set_index("ID_Producto")

,Nombre_producto,Categoría,Precio_Unitario,Stock
ID_Producto,,,,
1,Leche,Lácteos,"12,24",3327
2,Yogur,Lácteos,"5,21",3358
3,Queso cremoso,Lácteos,"17,23",3167
4,Queso rallado,Lácteos,"19,23",2099
5,Manteca,Lácteos,"5,65",4929
6,Asado,Carnicería,"28,56",5137
7,Chorizo,Carnicería,"11,25",4068
8,Milanesa,Carnicería,"16,21",3140
9,Pollo,Carnicería,"18,56",4051


In [15]:
# Cantidad de productos por cada categoria
print(productos['Categoría'].value_counts().sort_index())

Categoría
Bebidas                4
Carnicería             6
Congelados             4
Conservas              4
Frutas y Verduras      6
Galletitas y Snacks    4
Lácteos                5
Panadería              5
Name: count, dtype: int64


In [16]:
#Convirtiendo la columna 'Precio_Unitario' a float
productos['Precio_Unitario'] = productos['Precio_Unitario'].str.replace(',', '.').str.replace(' ','').astype(float)

Analizando el Excel se vio un problema:


Algunas fechas tienen formato: '31/01/2024' (DD/MM/YYYY), mientras que otras tienen: '31/1/2024' (D/M/YYYY), esto causa inconsistencia en la conversión.

In [17]:
# Convertir correctamente las fechas
ventas['Fecha'] = pd.to_datetime(ventas['Fecha'], dayfirst=True, errors='coerce')

# Mostrar el rango real corregido
fecha_min = ventas['Fecha'].min()
fecha_max = ventas['Fecha'].max()
print(f"Rango de fechas corregido: {fecha_min.strftime('%d/%m/%Y')} to {fecha_max.strftime('%d/%m/%Y')}")
print(f"Duracion real: {(fecha_max - fecha_min).days} dias")

Rango de fechas corregido: 31/01/2024 to 30/12/2024
Duracion real: 334 dias


In [18]:
# Métodos de pago
print(f"Métodos: {len(ventas['Método_Pago'].unique())}")
print(ventas['Método_Pago'].value_counts().sort_index())

Métodos: 5
Método_Pago
1    557
2    262
3    542
4    872
5    796
Name: count, dtype: int64


Significado de las 5 categorias de arriba

In [19]:
metodos_de_pago[['ID_Metodo', 'Método']].set_index("ID_Metodo")

,Método
ID_Metodo,
1,Efectivo
2,Tarjeta de Crédito
3,Tarjeta de Débito
4,Mercado Pago
5,Transferencia


In [20]:
# Análisis de estados
print("Estado de las transacciones: ")
print(ventas['Estado'].value_counts())

Estado de las transacciones: 
Estado
Completa     2548
Pendiente     471
Cancelada      10
Name: count, dtype: int64


In [21]:
# 5. Análisis de granularidad
transacciones_por_cliente = ventas.groupby('ID_Cliente').size()
print(f"Transacciones por cliente: {transacciones_por_cliente.min()}-{transacciones_por_cliente.max()}")
print(f"Promedio: {transacciones_por_cliente.mean():.1f} transacciones por cliente")

Transacciones por cliente: 2-19
Promedio: 9.3 transacciones por cliente


## Resumen Ejecutivo - Análisis de Datos

### Volúmenes y Unicidad
- ID_Venta: 3,000 valores únicos de 3,029 registros
- ID_Cliente: 326 clientes únicos
- ID_Producto: 38 productos únicos

### Distribución Geográfica
- Regiones: 6 regiones atendidas
  - Buenos Aires: 111 clientes
  - Patagonia: 64 clientes  
  - Centro: 63 clientes
  - Cuyo: 44 clientes
  - NEA: 37 clientes
  - NOA: 7 clientes

### Catálogo de Productos
- Categorías: 8 categorías de productos
  - Carnicería: 6 productos
  - Frutas y Verduras: 6 productos
  - Lácteos: 5 productos
  - Panadería: 5 productos
  - Bebidas: 4 productos
  - Congelados: 4 productos
  - Conservas: 4 productos
  - Galletitas y Snacks: 4 productos

### Período de Análisis
- Rango de fechas: 01/02/2024 a 30/12/2024
- Duración: 11 meses de datos

### Métodos de Pago
- Total métodos: 5 formas de pago
  - Mercado Pago: 872 transacciones
  - Transferencia: 796 transacciones
  - Efectivo: 557 transacciones
  - Tarjeta de Débito: 542 transacciones
  - Tarjeta de Crédito: 262 transacciones

### Estado de Transacciones
- Completa: 2,548 transacciones
- Pendiente: 471 transacciones 
- Cancelada: 10 transacciones 

### Comportamiento de Clientes
- Transacciones por cliente: 2-19 transacciones
- Promedio: 9.3 transacciones por cliente

In [22]:
# DETECCIÓN DE DUPLICADOS 
# Duplicados en ID_Venta
duplicados_id = ventas.duplicated(subset=['ID_Venta']).sum()
print(f"Duplicados en ID_Venta: {duplicados_id}")

# Duplicados semánticos (misma transacción)
duplicados_semanticos = ventas.duplicated(subset=['Fecha', 'ID_Cliente', 'ID_Producto', 'Cantidad']).sum()
print(f"Duplicados semánticos: {duplicados_semanticos}")

Duplicados en ID_Venta: 29
Duplicados semánticos: 29


In [23]:
# CONSTRUCCIÓN DE VARIABLES DERIVADAS
ventas_transformadas = ventas.copy()

# Variables temporales
ventas_transformadas['Semana'] = ventas_transformadas['Fecha'].dt.isocalendar().week
ventas_transformadas['Mes'] = ventas_transformadas['Fecha'].dt.month
ventas_transformadas['Dia_Semana'] = ventas_transformadas['Fecha'].dt.day_name()
ventas_transformadas['Trimestre'] = ventas_transformadas['Fecha'].dt.quarter

print(" Variables creadas: Semana, Mes, Dia_Semana, Trimestre")

 Variables creadas: Semana, Mes, Dia_Semana, Trimestre


In [24]:
# UNIÓN CON DATASET PRODUCTOS Y CÁLCULO DE VENTA TOTAL
# Unir todos los datos de productos en una sola operación
ventas_completo = ventas_transformadas.merge(
    productos[['ID_Producto', 'Nombre_producto', 'Categoría', 'Precio_Unitario', 'Stock']], 
    on='ID_Producto', 
    how='left'
)

In [25]:
# TRATAMIENTO DE DUPLICADOS

# Ver los duplicados específicos
duplicados_completos = ventas[ventas.duplicated(subset=['ID_Venta'], keep=False)]
print("\nEjemplo de duplicados:")
display(duplicados_completos.head(10))

# DECISIÓN: Eliminar duplicados exactos manteniendo el primero
registros_antes = len(ventas)
ventas = ventas.drop_duplicates(subset=['ID_Venta'], keep='first')
registros_despues = len(ventas)

print(f"RESULTADO:")
print(f"Registros antes: {registros_antes}")
print(f"Registros después: {registros_despues}")
print(f"Duplicados eliminados: {registros_antes - registros_despues}")

# Actualizar datasets derivados
ventas_transformadas = ventas.copy()
ventas_completo = ventas_completo.drop_duplicates(subset=['ID_Venta'], keep='first')


Ejemplo de duplicados:


,ID_Venta,Fecha,ID_Cliente,ID_Producto,Cantidad,Método_Pago,Estado
31,680,2024-02-04,112,37,2,5,Completa
36,680,2024-02-04,112,37,2,5,Completa
102,681,2024-02-13,14,29,3,3,Completa
111,681,2024-02-13,14,29,3,3,Completa
121,696,2024-02-15,76,13,3,1,Completa
129,696,2024-02-15,76,13,3,1,Completa
187,691,2024-02-21,175,14,1,1,Completa
194,691,2024-02-21,175,14,1,1,Completa
509,669,2024-03-28,44,19,4,3,Completa
514,669,2024-03-28,44,19,4,3,Completa


RESULTADO:
Registros antes: 3029
Registros después: 3000
Duplicados eliminados: 29


In [26]:
print("RANGOS DE LOS DATOS:")

for columna in ventas.columns:
    if ventas[columna].dtype in ['int64', 'float64']:
        min_val = ventas[columna].min()
        max_val = ventas[columna].max()
        print(f"• {columna}:")
        print(f"  Mínimo: {min_val}")
        print(f"  Máximo: {max_val}")
        print(f"  Rango: {min_val} a {max_val}")

RANGOS DE LOS DATOS:
• ID_Venta:
  Mínimo: 1
  Máximo: 3000
  Rango: 1 a 3000
• ID_Cliente:
  Mínimo: 1
  Máximo: 326
  Rango: 1 a 326
• ID_Producto:
  Mínimo: 1
  Máximo: 38
  Rango: 1 a 38
• Cantidad:
  Mínimo: 1
  Máximo: 6
  Rango: 1 a 6
• Método_Pago:
  Mínimo: 1
  Máximo: 5
  Rango: 1 a 5


In [27]:
# Calcular métricas por producto
ventas_completo['Venta_Total'] = ventas_completo['Cantidad'] * ventas_completo['Precio_Unitario']

In [28]:

resumen_productos = ventas_completo.groupby(['ID_Producto', 'Nombre_producto', 'Categoría', 'Precio_Unitario', 'Stock']).agg(
    Unidades_Vendidas=('Cantidad', 'sum'),
    Venta_Total=('Venta_Total', 'sum'),
    Transacciones=('ID_Venta', 'count')
).reset_index()

# Calcular totales
total_unidades = resumen_productos['Unidades_Vendidas'].sum()
total_ventas = resumen_productos['Venta_Total'].sum()

# Calcular porcentajes
resumen_productos['%_Unidades'] = (resumen_productos['Unidades_Vendidas'] / total_unidades) * 100
resumen_productos['%_Venta'] = (resumen_productos['Venta_Total'] / total_ventas) * 100

# Ordenar por venta total descendente
resumen_productos = resumen_productos.sort_values('Venta_Total', ascending=False)

print(f"Total unidades vendidas: {total_unidades:,}")
print(f"Total ventas: ${total_ventas:,.2f}")

# Mostrar resultados
print("TOP 10 PRODUCTOS POR VENTAS:")
print(f"{'Producto':<25} {'Categoría':<20} {'Unidades':<10} {'Venta Total':<12} {'% Und':<8} {'% Venta':<8} {'Transacc':<10}")

for _, row in resumen_productos.iterrows():
    print(f"{row['Nombre_producto'][:24]:<25} {row['Categoría'][:19]:<20} "
          f"{row['Unidades_Vendidas']:<10,} ${row['Venta_Total']:<11,.2f} "
          f"% {row['%_Unidades']:<7.1f} % {row['%_Venta']:<7.1f} {row['Transacciones']:<10,}")


Total unidades vendidas: 10,441
Total ventas: $103,103.19
TOP 10 PRODUCTOS POR VENTAS:
Producto                  Categoría            Unidades   Venta Total  % Und    % Venta  Transacc  
Asado                     Carnicería           299        $8,539.44    % 2.9     % 8.3     81        
Milanesa                  Carnicería           320        $5,187.20    % 3.1     % 5.0     89        
Pizza congelada           Congelados           332        $5,129.40    % 3.2     % 5.0     86        
Queso rallado             Lácteos              259        $4,980.57    % 2.5     % 4.8     77        
Queso cremoso             Lácteos              268        $4,617.64    % 2.6     % 4.5     84        
Pollo                     Carnicería           246        $4,565.76    % 2.4     % 4.4     71        
Cerveza                   Bebidas              350        $4,039.00    % 3.4     % 3.9     98        
Empanadas                 Congelados           277        $3,750.58    % 2.7     % 3.6     75      

In [29]:
counts_by_state = pd.crosstab(
    index=ventas['ID_Producto'], 
    columns=ventas['Estado']
)

In [30]:
resumen_productos = resumen_productos.join(counts_by_state, on='ID_Producto')

resumen_productos.head(5)

,ID_Producto,Nombre_producto,Categoría,Precio_Unitario,Stock,Unidades_Vendidas,Venta_Total,Transacciones,%_Unidades,%_Venta,Cancelada,Completa,Pendiente
5,6,Asado,Carnicería,28.56,5137,299,8539.44,81,2.863710,8.282421,0,71,10
7,8,Milanesa,Carnicería,16.21,3140,320,5187.20,89,3.064841,5.031076,0,76,13
24,25,Pizza congelada,Congelados,15.45,1640,332,5129.40,86,3.179772,4.975016,1,79,6
3,4,Queso rallado,Lácteos,19.23,2099,259,4980.57,77,2.480605,4.830665,1,67,9
2,3,Queso cremoso,Lácteos,17.23,3167,268,4617.64,84,2.566804,4.478659,0,66,18


In [31]:
resumen_productos.to_csv('../data_clean/resumen_productos.csv', index=False)

In [32]:
# Resumen por categoría
print("RESUMEN POR CATEGORÍA:")
resumen_categoria = resumen_productos.groupby('Categoría').agg(
    Unidades_Vendidas=('Unidades_Vendidas', 'sum'),
    Venta_Total=('Venta_Total', 'sum'),
    Productos=('ID_Producto', 'count')
).reset_index()

resumen_categoria['%_Unidades'] = (resumen_categoria['Unidades_Vendidas'] / total_unidades) * 100
resumen_categoria['%_Venta'] = (resumen_categoria['Venta_Total'] / total_ventas) * 100

# NUEVA COLUMNA: Venta promedio por producto
resumen_categoria['Venta_Por_Producto'] = resumen_categoria['Venta_Total'] / resumen_categoria['Productos']

resumen_categoria = resumen_categoria.sort_values('Venta_Total', ascending=False)

# Tabla con nombres de columnas
print(f"{'Categoría':<20} {'Unidades':<10} {'Venta Total':<12} {'% Und':<8} {'% Venta':<8} {'# Prod':<8} {'Vta/Prod':<12}")
for _, row in resumen_categoria.iterrows():
    print(f"{row['Categoría']:<20} {row['Unidades_Vendidas']:<10,} ${row['Venta_Total']:<11,.2f} "
          f"{row['%_Unidades']:<7.1f}% {row['%_Venta']:<7.1f}% {row['Productos']:<8} ${row['Venta_Por_Producto']:<11,.2f}")

RESUMEN POR CATEGORÍA:
Categoría            Unidades   Venta Total  % Und    % Venta  # Prod   Vta/Prod    
Carnicería           1,655      $27,924.82   15.9   % 27.1   % 6        $4,654.14   
Lácteos              1,376      $16,093.78   13.2   % 15.6   % 5        $3,218.76   
Congelados           1,283      $14,918.25   12.3   % 14.5   % 4        $3,729.56   
Panadería            1,258      $12,678.59   12.0   % 12.3   % 5        $2,535.72   
Bebidas              1,176      $11,673.71   11.3   % 11.3   % 4        $2,918.43   
Frutas y Verduras    1,465      $7,820.73    14.0   % 7.6    % 6        $1,303.46   
Galletitas y Snacks  1,141      $7,129.08    10.9   % 6.9    % 4        $1,782.27   
Conservas            1,087      $4,864.23    10.4   % 4.7    % 4        $1,216.06   


In [33]:
# ANÁLISIS DE INVENTARIO Y ROTACIÓN
dias_totales = (ventas_completo['Fecha'].max() - ventas_completo['Fecha'].min()).days + 1

# Calcular promedio diario de ventas por producto
ventas_diarias_promedio = ventas_completo[ventas_completo['Estado'] == 'Completa'].groupby(
    ['ID_Producto', 'Nombre_producto']
).agg({
    'Cantidad': 'sum'
}).reset_index()

ventas_diarias_promedio['Promedio_Diario_Unidades'] = ventas_diarias_promedio['Cantidad'] / dias_totales

# Unir con datos de stock
analisis_inventario = ventas_diarias_promedio.merge(
    productos[['ID_Producto', 'Stock', 'Precio_Unitario']],
    on='ID_Producto',
    how='left'
)

In [34]:

# Calcular métricas adicionales
analisis_inventario['Dinero_en_Stock'] = analisis_inventario['Stock'] * analisis_inventario['Precio_Unitario']
analisis_inventario['Dias_de_Inventario'] = analisis_inventario['Stock'] / analisis_inventario['Promedio_Diario_Unidades']

# Seleccionar y ordenar columnas exactamente como pediste
analisis_inventario = analisis_inventario[[
    'Nombre_producto', 
    'Promedio_Diario_Unidades', 
    'Stock', 
    'Dinero_en_Stock', 
    'Dias_de_Inventario'
]]

In [35]:

# Mostrar tabla simple
print("TABLA DE ANÁLISIS DE INVENTARIO:")
print(f"{'Producto':<25} {'Promedio Diario':<15} {'Stock':<10} {'Dinero en Stock':<15} {'Días Inventario':<15}")
print("-" * 88)

for _, row in analisis_inventario.iterrows():
    print(f"{row['Nombre_producto'][:24]:<25} {row['Promedio_Diario_Unidades']:<15.2f} {row['Stock']:<10} ${row['Dinero_en_Stock']:<14.2f} {row['Dias_de_Inventario']:<15.1f}")

# Total de dinero inmovilizado en inventario
total_dinero_stock = analisis_inventario['Dinero_en_Stock'].sum()
print(f"\nTOTAL DE DINERO INMOVILIZADO EN STOCK: ${total_dinero_stock:,.2f}")

TABLA DE ANÁLISIS DE INVENTARIO:
Producto                  Promedio Diario Stock      Dinero en Stock Días Inventario
----------------------------------------------------------------------------------------
Leche                     0.69            3327       $40722.48       4804.1         
Yogur                     0.65            3358       $17495.18       5160.2         
Queso cremoso             0.63            3167       $54567.41       5052.1         
Queso rallado             0.69            2099       $40363.77       3044.0         
Manteca                   0.71            4929       $27848.85       6967.2         
Asado                     0.79            5137       $146712.72      6469.5         
Chorizo                   0.69            4068       $45765.00       5925.1         
Milanesa                  0.79            3140       $50899.40       3984.5         
Pollo                     0.64            4051       $75186.56       6371.3         
Costilla de cerdo         0.

In [36]:
# ANÁLISIS RFM (Recency, Frequency, Monetary)
fecha_referencia = ventas_completo['Fecha'].max() + pd.Timedelta(days=1)

# Filtrar solo transacciones completas para RFM
ventas_completas = ventas_completo[ventas_completo['Estado'] == 'Completa']

# Calcular métricas RFM por cliente
rfm_data = ventas_completas.groupby('ID_Cliente').agg({
    'Fecha': lambda x: (fecha_referencia - x.max()).days,  # Recency
    'ID_Venta': 'count',                                   # Frequency  
    'Venta_Total': 'sum'                                   # Monetary
}).reset_index()

rfm_data.columns = ['ID_Cliente', 'Recency', 'Frequency', 'Monetary']
# Mostrar estadísticas RFM
print("Estadísticas RFM:")
print(f"Recency (días desde última compra): {rfm_data['Recency'].mean():.1f} días en promedio")
print(f"Frequency (compras por cliente): {rfm_data['Frequency'].mean():.1f} compras en promedio")
print(f"Monetary (gasto por cliente): ${rfm_data['Monetary'].mean():.2f} en promedio")

Estadísticas RFM:
Recency (días desde última compra): 42.5 días en promedio
Frequency (compras por cliente): 7.7 compras en promedio
Monetary (gasto por cliente): $269.11 en promedio


In [37]:
# SEGMENTACIÓN RFM
# Crear segmentos RFM (usando cuartiles)
rfm_data['R_Score'] = pd.qcut(rfm_data['Recency'], 4, labels=[4, 3, 2, 1])  # Menor Recency = mejor
rfm_data['F_Score'] = pd.qcut(rfm_data['Frequency'], 4, labels=[1, 2, 3, 4])  # Mayor Frequency = mejor
rfm_data['M_Score'] = pd.qcut(rfm_data['Monetary'], 4, labels=[1, 2, 3, 4])   # Mayor Monetary = mejor

# Combinar scores
rfm_data['RFM_Score'] = rfm_data['R_Score'].astype(str) + rfm_data['F_Score'].astype(str) + rfm_data['M_Score'].astype(str)

# Crear segmentos basados en RFM
def asignar_segmento(rfm_score):
    r, f, m = int(rfm_score[0]), int(rfm_score[1]), int(rfm_score[2])
    if r == 4 and f == 4 and m == 4:
        return 'Campeones'
    elif r == 4 and f >= 3:
        return 'Clientes Leales'
    elif r >= 3:
        return 'Clientes con Potencial'
    elif r == 2:
        return 'Clientes en Riesgo'
    else:
        return 'Clientes Dormidos'

rfm_data['Segmento'] = rfm_data['RFM_Score'].apply(asignar_segmento)

# Mostrar distribución de segmentos
print("\nDistribución de segmentos de clientes:")
segmentos_dist = rfm_data['Segmento'].value_counts()
for segmento, count in segmentos_dist.items():
    porcentaje = (count / len(rfm_data)) * 100
    print(f"  {segmento}: {count} clientes ({porcentaje:.1f}%)")


Distribución de segmentos de clientes:
  Clientes con Potencial: 125 clientes (38.3%)
  Clientes en Riesgo: 82 clientes (25.2%)
  Clientes Dormidos: 81 clientes (24.8%)
  Campeones: 19 clientes (5.8%)
  Clientes Leales: 19 clientes (5.8%)


### Segmentación RFM de Clientes

Campeones (5.8%) - compran frecuentemente, recientemente y gastan mucho.


Clientes Leales (6.1%) - Compran regularmente y gastan bien, pero no tan recientemente. Estrategia: Programas de recompensas y ofertas exclusivas.


Clientes Potenciales (38.0%) - Compraron recientemente pero con menor frecuencia/gasto. Oportunidad de crecimiento.



Clientes en Riesgo (25.2%) - Antes eran buenos clientes pero no han comprado recientemente. 


Clientes Dormidos (24.8%) - No han comprado en mucho tiempo y tenían baja participación. 

In [38]:
# ANÁLISIS DE ESTACIONALIDAD Y PATRONES TEMPORALES

# Ventas por mes
ventas_mensuales = ventas_completo[ventas_completo['Estado'] == 'Completa'].groupby('Mes').agg({
    'Venta_Total': 'sum',
    'ID_Venta': 'count',
    'ID_Cliente': 'nunique'
}).reset_index()

ventas_mensuales.columns = ['Mes', 'Ventas_Totales', 'Transacciones', 'Clientes_Unicos']

print("VENTAS MENSUALES:")
print(f"{'Mes':<8} {'Ventas':<12} {'Transacc':<10} {'Clientes':<10} {'Ticket Prom':<12}")

for _, row in ventas_mensuales.iterrows():
    ticket_promedio = row['Ventas_Totales'] / row['Transacciones'] if row['Transacciones'] > 0 else 0
    print(f"{row['Mes']:<8} ${row['Ventas_Totales']:<11,.2f} {row['Transacciones']:<10} {row['Clientes_Unicos']:<10} ${ticket_promedio:<11.2f}")

VENTAS MENSUALES:
Mes      Ventas       Transacc   Clientes   Ticket Prom 
1.0      $255.98      7.0        7.0        $36.57      
2.0      $7,370.28    210.0      152.0      $35.10      
3.0      $8,837.72    235.0      173.0      $37.61      
4.0      $7,639.11    215.0      159.0      $35.53      
5.0      $7,459.69    225.0      164.0      $33.15      
6.0      $8,605.66    263.0      179.0      $32.72      
7.0      $7,677.98    230.0      167.0      $33.38      
8.0      $8,055.87    232.0      172.0      $34.72      
9.0      $8,210.48    227.0      169.0      $36.17      
10.0     $7,783.32    240.0      168.0      $32.43      
11.0     $6,965.29    211.0      150.0      $33.01      
12.0     $8,868.51    228.0      174.0      $38.90      


In [39]:
# Ventas por día de la semana
print("VENTAS POR DÍA DE LA SEMANA:")
ventas_diarias = ventas_completo[ventas_completo['Estado'] == 'Completa'].groupby('Dia_Semana').agg({
    'Venta_Total': 'sum',
    'ID_Venta': 'count',
    'ID_Cliente': 'nunique'
}).reset_index()

# Ordenar días de la semana lógico
dias_orden = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
ventas_diarias['Dia_Semana'] = pd.Categorical(ventas_diarias['Dia_Semana'], categories=dias_orden, ordered=True)
ventas_diarias = ventas_diarias.sort_values('Dia_Semana')

print(f"{'Día':<12} {'Ventas':<12} {'Transacc':<10} {'Clientes':<10} {'Ticket Prom':<12}")

for _, row in ventas_diarias.iterrows():
    ticket_promedio = row['Venta_Total'] / row['ID_Venta'] if row['ID_Venta'] > 0 else 0
    dia_es = {'Monday': 'Lunes', 'Tuesday': 'Martes', 'Wednesday': 'Miércoles', 
              'Thursday': 'Jueves', 'Friday': 'Viernes', 'Saturday': 'Sábado', 'Sunday': 'Domingo'}
    print(f"{dia_es[row['Dia_Semana']]:<12} ${row['Venta_Total']:<11,.2f} {row['ID_Venta']:<10} {row['ID_Cliente']:<10} ${ticket_promedio:<11.2f}")

VENTAS POR DÍA DE LA SEMANA:
Día          Ventas       Transacc   Clientes   Ticket Prom 
Lunes        $12,175.43   345        211        $35.29      
Martes       $12,734.74   339        210        $37.57      
Miércoles    $12,586.29   369        224        $34.11      
Jueves       $12,779.89   381        223        $33.54      
Viernes      $13,129.52   358        212        $36.67      
Sábado       $11,682.02   347        216        $33.67      
Domingo      $12,642.00   384        226        $32.92      


## Bitácora de Decisiones

### Valores Nulos
No se detectaron valores nulos en ningún dataset, no se requirió imputación

### Duplicados
29 registros duplicados en ID_Venta (0.96% del total), ID_Venta debe ser único por definición de llave primaria, se eliminaron 29 registros.

### Outliers
No se detectaron outliers, todos los valores dentro de rangos esperados

### Normalización/Estandarización
No se aplicó normalización, escalas interpretables naturalmente


## Variables Derivadas Construidas

### Temporales
- Semana (ISO), Mes, Día_Semana, Trimestre
- Por qué: Análisis de estacionalidad y patrones temporales

### Financieras
-  Venta_Total = Cantidad × Precio_Unitario
- Por qué: Cálculo de valor monetario por transacción
- Cómo: Unión con dataset productos y multiplicación
- Resultado: Base para análisis RFM y métricas de revenue

### RFM
- Segmentación de clientes por comportamiento
- Cómo: Agrupación por ID_Cliente y cálculo de métricas
- Resultado: Dataset listo para segmentación RFM

In [40]:
# DISTRIBUCIÓN DE VARIABLES NUMÉRICAS
# Subplots interactivos
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=('Distribución de Cantidad de productos por Transacción', 
                    'Distribución de Venta por Transacción',
                    'Distribución de Precios Unitarios',
                    'Venta Total por Categoría (Top 5)'),
    specs=[[{"type": "histogram"}, {"type": "histogram"}],
           [{"type": "histogram"}, {"type": "box"}]]
)

# Cantidad
fig.add_trace(go.Histogram(x=ventas_completo['Cantidad'], nbinsx=6, name='Cantidad'), row=1, col=1)

# Venta Total
fig.add_trace(go.Histogram(x=ventas_completo['Venta_Total'], name='Venta Total'), row=1, col=2)

# Precio Unitario
fig.add_trace(go.Histogram(x=ventas_completo['Precio_Unitario'], name='Precio Unitario'), row=2, col=1)

# Boxplot por categoría (top 5)
top_categorias = resumen_categoria.head(5)['Categoría'].tolist()
ventas_top_cat = ventas_completo[ventas_completo['Categoría'].isin(top_categorias)]

for categoria in top_categorias:
    data_cat = ventas_top_cat[ventas_top_cat['Categoría'] == categoria]['Venta_Total']
    fig.add_trace(go.Box(y=data_cat, name=categoria, showlegend=False), row=2, col=2)

fig.update_layout(height=800, title_text="Distribuciones de Variables Numéricas")
fig.show()

## Interpretación
### Grafica 1 Distribución de Cantidad de productos por Transacción
Los clientes no tienen una tendencia clara en la cantidad de productos que compran


### Grafica 2 Distribución de Precios Unitarios
Los clientes tienden a tener un monto de compra chico

### Grafica 3 Distribución de Precios Unitarios
Muestra la cantidad de veces que se compro un producto en este rango de precios

### Grafica 4 Venta Total por Categoría (Top 5)
La linea de en medio es el promedio de transaccion en la venta, maximo, minimo de ventas y el rango normal de estas.

In [41]:
# DISTRIBUCIÓN DE VENTAS POR CATEGORÍA
# Gráfico de barras interactivo
fig1 = px.bar(resumen_categoria, 
              x='Venta_Total', 
              y='Categoría',
              orientation='h',
              title='Ventas Totales por Categoría',
              color='Venta_Total',
              color_continuous_scale='Viridis',
              text='Venta_Total')

fig1.update_layout(yaxis={'categoryorder':'total ascending'},
                   xaxis_title='Ventas Totales ($)',
                   yaxis_title='',
                   showlegend=False)
fig1.show()

In [42]:
# Gráfico de torta interactivo
fig2 = px.pie(resumen_categoria,
              values='Venta_Total',
              names='Categoría',
              title='Distribución Porcentual de Ventas por Categoría',
              hole=0.3)

fig2.update_traces(textposition='inside', textinfo='percent+label')
fig2.show()

In [43]:
# DISTRIBUCIÓN DE UNIDADES VENDIDAS POR CATEGORÍA

# Gráfico de barras interactivo - UNIDADES
fig1 = px.bar(resumen_categoria, 
              x='Unidades_Vendidas', 
              y='Categoría',
              orientation='h',
              title='Unidades Vendidas por Categoría',
              color='Unidades_Vendidas',
              color_continuous_scale='Blues',
              text='Unidades_Vendidas')

fig1.update_layout(yaxis={'categoryorder':'total ascending'},
                   xaxis_title='Total de Unidades Vendidas',
                   yaxis_title='',
                   showlegend=False)
fig1.show()

In [44]:
# Gráfico de torta interactivo - UNIDADES
fig2 = px.pie(resumen_categoria,
              values='Unidades_Vendidas',
              names='Categoría',
              title='Distribución Porcentual de Unidades Vendidas por Categoría',
              hole=0.3)

fig2.update_traces(textposition='inside', textinfo='percent+label')
fig2.show()

In [45]:
# HISTOGRAMA de ventas mensuales
# Preparar datos con nombres de meses
ventas_mensual_categoria = ventas_completo[ventas_completo['Estado'] == 'Completa'].groupby(
    ['Mes', 'Categoría']
).agg({'Cantidad': 'sum'}).reset_index()

# Mapear números de mes a nombres
nombres_meses = {
    1: 'Enero', 2: 'Febrero', 3: 'Marzo', 4: 'Abril', 5: 'Mayo', 6: 'Junio',
    7: 'Julio', 8: 'Agosto', 9: 'Septiembre', 10: 'Octubre', 11: 'Noviembre', 12: 'Diciembre'
}

ventas_mensual_categoria['Mes_Nombre'] = ventas_mensual_categoria['Mes'].map(nombres_meses)

# Asegurar orden correcto de meses
ventas_mensual_categoria['Mes_Nombre'] = pd.Categorical(
    ventas_mensual_categoria['Mes_Nombre'], 
    categories=list(nombres_meses.values()), 
    ordered=True
)

# Crear histograma apilado
fig = px.histogram(ventas_mensual_categoria,
                   x='Mes_Nombre',
                   y='Cantidad',
                   color='Categoría',
                   barmode='stack',
                   title='Distribución de Ventas Mensuales por Categoría',
                   labels={'Venta_Total': 'Ventas Acumuladas ($)', 'Mes_Nombre': 'Mes'},
                   height=600,
                   text_auto='.2s')  # Mostrar valores en las barras

fig.update_layout(
    xaxis_title='Mes',
    yaxis_title='Ventas Acumuladas (U)',
    bargap=0.15,
    showlegend=True
)

# Rotar etiquetas del eje X para mejor legibilidad
fig.update_xaxes(tickangle=45)

fig.show()

In [46]:
# Preparar datos específicos para ventas en dinero
ventas_mensual_dinero = ventas_completo[ventas_completo['Estado'] == 'Completa'].groupby(
    ['Mes', 'Categoría']
).agg({'Venta_Total': 'sum'}).reset_index()

# Mapear números de mes a nombres
nombres_meses = {
    1: 'Enero', 2: 'Febrero', 3: 'Marzo', 4: 'Abril', 5: 'Mayo', 6: 'Junio',
    7: 'Julio', 8: 'Agosto', 9: 'Septiembre', 10: 'Octubre', 11: 'Noviembre', 12: 'Diciembre'
}

ventas_mensual_dinero['Mes_Nombre'] = ventas_mensual_dinero['Mes'].map(nombres_meses)

# Asegurar orden correcto de meses
ventas_mensual_dinero['Mes_Nombre'] = pd.Categorical(
    ventas_mensual_dinero['Mes_Nombre'], 
    categories=list(nombres_meses.values()), 
    ordered=True
)

# Calcular totales por mes - DINERO
totales_mes_dinero = ventas_mensual_dinero.groupby('Mes_Nombre')['Venta_Total'].sum().reset_index()

# Crear tabla interactiva - DINERO
fig_tabla_dinero = go.Figure(data=[go.Table(
    header=dict(
        values=['<b>Mes</b>', '<b>Ventas Totales</b>'],
        fill_color='lightgreen',
        align='left',
        font=dict(size=14, color='black')
    ),
    cells=dict(
        values=[
            totales_mes_dinero['Mes_Nombre'],
            ['$' + f'{x:,.2f}' for x in totales_mes_dinero['Venta_Total']]
        ],
        fill_color='lightyellow',
        align='left',
        font=dict(size=12)
    )
)])

# Gráfico de barras para ventas por mes - DINERO
fig_barras_dinero = px.bar(totales_mes_dinero,
                    x='Mes_Nombre',
                    y='Venta_Total',
                    title='Ventas Totales por Mes (Dinero)',
                    labels={'Venta_Total': 'Ventas Totales ($)', 'Mes_Nombre': 'Mes'},
                    text_auto='$.2s')

fig_barras_dinero.update_layout(xaxis_tickangle=45)
fig_barras_dinero.update_traces(marker_color='seagreen')
fig_barras_dinero.show()

C:\Users\Dania\AppData\Local\Temp\ipykernel_23332\2631255546.py:22: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.



In [47]:
# TABLA DE TOP 5 Y BOTTOM 5 PRODUCTOS POR UNIDADES VENDIDAS
# Obtener top 5 productos más vendidos (por unidades)
top_5_unidades = resumen_productos.nlargest(5, 'Unidades_Vendidas')[['Nombre_producto', 'Categoría', 'Unidades_Vendidas', 'Venta_Total']]

# Obtener bottom 5 productos menos vendidos (por unidades)
bottom_5_unidades = resumen_productos.nsmallest(5, 'Unidades_Vendidas')[['Nombre_producto', 'Categoría', 'Unidades_Vendidas', 'Venta_Total']]

# Crear tabla combinada
tabla_comparativa = pd.concat([
    top_5_unidades.assign(Tipo='TOP 5'),
    bottom_5_unidades.assign(Tipo='BOTTOM 5')
], ignore_index=True)

# Crear tabla interactiva con Plotly
fig_tabla = go.Figure(data=[go.Table(
    header=dict(
        values=['<b>Tipo</b>', '<b>Producto</b>', '<b>Categoría</b>', '<b>Unidades Vendidas</b>', '<b>Ventas Totales</b>'],
        fill_color='lightblue',
        align='left',
        font=dict(size=14, color='black')
    ),
    cells=dict(
        values=[
            tabla_comparativa['Tipo'],
            tabla_comparativa['Nombre_producto'],
            tabla_comparativa['Categoría'],
            [f'{x:,}' for x in tabla_comparativa['Unidades_Vendidas']],
            ['$' + f'{x:,.2f}' for x in tabla_comparativa['Venta_Total']]
        ],
        fill_color=[['lightgreen' if tipo == 'TOP 5' else 'lightcoral' for tipo in tabla_comparativa['Tipo']]],
        align='left',
        font=dict(size=12)
    )
)])

fig_tabla.update_layout(
    title='Top 5 y Bottom 5 Productos por Unidades Vendidas',
    title_x=0.5,
    width=900,
    height=500
)

fig_tabla.show()

In [48]:
# Obtener los nombres de los productos top
top_5_nombres = top_5_unidades['Nombre_producto'].tolist()

# Crear dataframe con las ventas diarias de los top 5 productos
ventas_diarias_top5 = (ventas_completo[
    (ventas_completo['Nombre_producto'].isin(top_5_nombres)) & 
    (ventas_completo['Estado'] == 'Completa')
].groupby(['Fecha', 'Nombre_producto'])['Cantidad']
 .sum()
 .reset_index())

# GRÁFICO 1: Comportamiento diario a lo largo del año
fig1 = px.line(ventas_diarias_top5, 
              x='Fecha', 
              y='Cantidad', 
              color='Nombre_producto',
              title='Comportamiento Diario de los 5 Productos Más Vendidos',
              labels={'Cantidad': 'Unidades Vendidas', 'Fecha': 'Fecha'},
              color_discrete_sequence=px.colors.qualitative.Bold)

fig1.update_layout(
    width=1200,
    height=600,
    xaxis=dict(tickformat='%b %Y', tickangle=45)
)

fig1.show()

# GRÁFICO 2: Patrón por día de la semana
ventas_diarias_top5['Dia_Semana'] = ventas_diarias_top5['Fecha'].dt.day_name()
ventas_por_dia_top5 = (ventas_diarias_top5.groupby(['Dia_Semana', 'Nombre_producto'])['Cantidad']
                       .mean()
                       .reset_index())

# Ordenar días de la semana
orden_dias = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
ventas_por_dia_top5['Dia_Semana'] = pd.Categorical(ventas_por_dia_top5['Dia_Semana'], 
                                                   categories=orden_dias, 
                                                   ordered=True)
ventas_por_dia_top5 = ventas_por_dia_top5.sort_values('Dia_Semana')

# Traducir a español
mapeo_dias_grafico = {
    'Monday': 'Lunes', 'Tuesday': 'Martes', 'Wednesday': 'Miércoles', 
    'Thursday': 'Jueves', 'Friday': 'Viernes', 'Saturday': 'Sábado', 'Sunday': 'Domingo'
}
ventas_por_dia_top5['Dia_Semana_ES'] = ventas_por_dia_top5['Dia_Semana'].map(mapeo_dias_grafico)

fig2 = px.line(ventas_por_dia_top5, 
               x='Dia_Semana_ES', 
               y='Cantidad', 
               color='Nombre_producto',
               title='Comportamiento por Día de la Semana - Top 5 Productos',
               labels={'Cantidad': 'Promedio Unidades Vendidas', 'Dia_Semana_ES': 'Día de la Semana'},
               color_discrete_sequence=px.colors.qualitative.Dark2)

fig2.update_layout(
    width=1000,
    height=500
)

fig2.show()

# GRÁFICO 3: Ventas acumuladas a lo largo del año
ventas_acumuladas = ventas_diarias_top5.sort_values(['Nombre_producto', 'Fecha'])
ventas_acumuladas['Ventas_Acumuladas'] = ventas_acumuladas.groupby('Nombre_producto')['Cantidad'].cumsum()

fig3 = px.line(ventas_acumuladas, 
               x='Fecha', 
               y='Ventas_Acumuladas', 
               color='Nombre_producto',
               title='Ventas Acumuladas a lo Largo del Año - Top 5 Productos',
               labels={'Ventas_Acumuladas': 'Unidades Acumuladas', 'Fecha': 'Fecha'})

fig3.update_layout(
    width=1200,
    height=600
)

fig3.show()

In [49]:
# ANÁLISIS COMBINADO VENTAS POR DÍA DE LA SEMANA Y MES

# Preparar datos: ventas por día de semana y mes
ventas_dia_mes = ventas_completo[ventas_completo['Estado'] == 'Completa'].copy()

# Mapear días de semana a español y ordenarlos
dias_ingles_espanol = {
    'Monday': 'Lunes',
    'Tuesday': 'Martes', 
    'Wednesday': 'Miércoles',
    'Thursday': 'Jueves',
    'Friday': 'Viernes',
    'Saturday': 'Sábado',
    'Sunday': 'Domingo'
}

ventas_dia_mes['Dia_Semana'] = ventas_dia_mes['Dia_Semana'].map(dias_ingles_espanol)

# Crear matriz para heatmap
heatmap_data = ventas_dia_mes.groupby(['Mes', 'Dia_Semana']).agg({
    'Venta_Total': 'sum'
}).reset_index()

# Pivotar para crear matriz
pivot_heatmap = heatmap_data.pivot(index='Dia_Semana', columns='Mes', values='Venta_Total')

# Ordenar días de semana lógicamente
orden_dias = ['Lunes', 'Martes', 'Miércoles', 'Jueves', 'Viernes', 'Sábado', 'Domingo']
pivot_heatmap = pivot_heatmap.reindex(orden_dias)

# Mapear números de mes a nombres
nombres_meses = {
    1: 'Ene', 2: 'Feb', 3: 'Mar', 4: 'Abr', 5: 'May', 6: 'Jun',
    7: 'Jul', 8: 'Ago', 9: 'Sep', 10: 'Oct', 11: 'Nov', 12: 'Dic'
}
pivot_heatmap.columns = [nombres_meses[mes] for mes in pivot_heatmap.columns]

# Crear heatmap interactivo
fig_heatmap = px.imshow(
    pivot_heatmap,
    title='Heatmap: Ventas por Día de la Semana y Mes',
    color_continuous_scale='Viridis',
    aspect='auto',
    labels=dict(x="Mes", y="Día de la Semana", color="Ventas ($)")
)

fig_heatmap.update_layout(
    width=800,
    height=500,
    xaxis_title='Mes',
    yaxis_title='Día de la Semana'
)

fig_heatmap.show()

In [50]:
# ANÁLISIS DE PRODUCTOS ESPECÍFICOS POR DÍA Y MES
print("PRODUCTOS MÁS VENDIDOS - MARTES DE JULIO Y DOMINGOS DE SEPTIEMBRE")

# Filtrar datos para martes de julio
martes_julio = ventas_completo[
    (ventas_completo['Mes'] == 7) & 
    (ventas_completo['Dia_Semana'] == 'Tuesday') &
    (ventas_completo['Estado'] == 'Completa')
]

# Filtrar datos para domingos de septiembre
domingos_septiembre = ventas_completo[
    (ventas_completo['Mes'] == 9) & 
    (ventas_completo['Dia_Semana'] == 'Sunday') &
    (ventas_completo['Estado'] == 'Completa')
]

# Análisis para martes de julio

productos_martes_julio = martes_julio.groupby(['ID_Producto', 'Nombre_producto', 'Categoría']).agg({
    'Cantidad': 'sum',
    'Venta_Total': 'sum',
    'ID_Venta': 'count'
}).reset_index()
    
productos_martes_julio = productos_martes_julio.sort_values('Cantidad', ascending=False)
top_martes_julio = productos_martes_julio.head(1)
    
print(" MARTES DE JULIO - PRODUCTO MÁS VENDIDO:")
print(f"Producto: {top_martes_julio['Nombre_producto'].iloc[0]}")
print(f"Unidades vendidas: {top_martes_julio['Cantidad'].iloc[0]:,}")
print(f"Venta total: ${top_martes_julio['Venta_Total'].iloc[0]:,.2f}")
print(f"Transacciones: {top_martes_julio['ID_Venta'].iloc[0]}")
    
# Top 5 productos de martes de julio
print(f"TOP 5 PRODUCTOS - MARTES DE JULIO:")
for i, (_, row) in enumerate(productos_martes_julio.head(5).iterrows(), 1):
    print(f"{i}. {row['Nombre_producto']} - {row['Cantidad']:,} unidades - ${row['Venta_Total']:,.2f}")


# Análisis para domingos de septiembre

productos_domingos_sept = domingos_septiembre.groupby(['ID_Producto', 'Nombre_producto', 'Categoría']).agg({
    'Cantidad': 'sum',
    'Venta_Total': 'sum',
    'ID_Venta': 'count'
}).reset_index()
    
productos_domingos_sept = productos_domingos_sept.sort_values('Cantidad', ascending=False)
top_domingos_sept = productos_domingos_sept.head(1)
    
print("DOMINGOS DE SEPTIEMBRE - PRODUCTO MÁS VENDIDO:")
print(f"Producto: {top_domingos_sept['Nombre_producto'].iloc[0]}")
print(f"Unidades vendidas: {top_domingos_sept['Cantidad'].iloc[0]:,}")
print(f"Venta total: ${top_domingos_sept['Venta_Total'].iloc[0]:,.2f}")
print(f"Transacciones: {top_domingos_sept['ID_Venta'].iloc[0]}")
    
# Top 5 productos de domingos de septiembre
print(f"TOP 5 PRODUCTOS - DOMINGOS DE SEPTIEMBRE:")
for i, (_, row) in enumerate(productos_domingos_sept.head(5).iterrows(), 1):
    print(f"{i}. {row['Nombre_producto']} - {row['Cantidad']:,} unidades - ${row['Venta_Total']:,.2f}")

PRODUCTOS MÁS VENDIDOS - MARTES DE JULIO Y DOMINGOS DE SEPTIEMBRE
 MARTES DE JULIO - PRODUCTO MÁS VENDIDO:
Producto: Arroz
Unidades vendidas: 15
Venta total: $63.75
Transacciones: 3
TOP 5 PRODUCTOS - MARTES DE JULIO:
1. Arroz - 15 unidades - $63.75
2. Pizza congelada - 14 unidades - $216.30
3. Cerveza - 13 unidades - $150.02
4. Manzanas - 12 unidades - $78.48
5. Leche - 12 unidades - $146.88
DOMINGOS DE SEPTIEMBRE - PRODUCTO MÁS VENDIDO:
Producto: Milanesa
Unidades vendidas: 18
Venta total: $291.78
Transacciones: 4
TOP 5 PRODUCTOS - DOMINGOS DE SEPTIEMBRE:
1. Milanesa - 18 unidades - $291.78
2. Tortas - 18 unidades - $274.14
3. Hamburgesas congeladas - 16 unidades - $152.64
4. Costilla de cerdo - 12 unidades - $171.00
5. Maníes - 9 unidades - $49.05


In [51]:
# Tabla resumen de promedios por día de semana
promedio_dia = ventas_dia_mes.groupby('Dia_Semana')['Venta_Total'].mean().reindex(orden_dias).reset_index()

fig_tabla_dias = go.Figure(data=[go.Table(
    header=dict(
        values=['<b>Día de la Semana</b>', '<b>Venta Promedio por Transacción</b>'],
        fill_color='lightgreen',
        align='left',
        font=dict(size=14)
    ),
    cells=dict(
        values=[
            promedio_dia['Dia_Semana'],
            ['$' + f'{x:,.2f}' for x in promedio_dia['Venta_Total']]
        ],
        fill_color='lightyellow',
        align='left'
    )
)])

fig_tabla_dias.update_layout(
    title='Venta Promedio por Día de la Semana',
    width=600,
    height=400
)

fig_tabla_dias.show()

In [52]:
# ANÁLISIS ADICIONAL: PRODUCTOS MÁS VENDIDOS LOS MARTES EN GENERAL
print("PRODUCTOS MÁS VENDIDOS - TODOS LOS MARTES (GENERAL)")

# Filtrar todos los martes
todos_martes = ventas_completo[
    (ventas_completo['Dia_Semana'] == 'Tuesday') &
    (ventas_completo['Estado'] == 'Completa')
]

productos_todos_martes = todos_martes.groupby(['ID_Producto', 'Nombre_producto', 'Categoría']).agg({
    'Cantidad': 'sum',
    'Venta_Total': 'sum',
    'ID_Venta': 'count'
}).reset_index()
    
productos_todos_martes = productos_todos_martes.sort_values('Cantidad', ascending=False)
top_todos_martes = productos_todos_martes.head(1)
    
print("TODOS LOS MARTES - PRODUCTO MÁS VENDIDO:")
print(f"Producto: {top_todos_martes['Nombre_producto'].iloc[0]}")
print(f"Unidades vendidas: {top_todos_martes['Cantidad'].iloc[0]:,}")
print(f"Venta total: ${top_todos_martes['Venta_Total'].iloc[0]:,.2f}")
print(f"Transacciones: {top_todos_martes['ID_Venta'].iloc[0]}")
    
# Top 5 productos de todos los martes
print("TOP 5 PRODUCTOS - TODOS LOS MARTES:")
for i, (_, row) in enumerate(productos_todos_martes.head(5).iterrows(), 1):
    print(f"{i}. {row['Nombre_producto']} - {row['Cantidad']:,} unidades - ${row['Venta_Total']:,.2f}")

PRODUCTOS MÁS VENDIDOS - TODOS LOS MARTES (GENERAL)
TODOS LOS MARTES - PRODUCTO MÁS VENDIDO:
Producto: Hamburgesas congeladas
Unidades vendidas: 66
Venta total: $629.64
Transacciones: 16
TOP 5 PRODUCTOS - TODOS LOS MARTES:
1. Hamburgesas congeladas - 66 unidades - $629.64
2. Cerveza - 55 unidades - $634.70
3. Pizza congelada - 53 unidades - $818.85
4. Atún enlatado - 51 unidades - $316.71
5. Arroz - 49 unidades - $208.25


In [53]:
# Crear el dataframe completo con todas las combinaciones fecha × categoría
fechas = ventas_completo[ventas_completo["Estado"] == "Completa"]["Fecha"].unique()
categorias = ventas_completo["Categoría"].unique()
todas_combinaciones = pd.DataFrame(list(itertools.product(fechas, categorias)),
                                   columns=["Fecha", "Categoría"])

# Sumar las cantidades reales vendidas por fecha y categoría
ventas_por_dia_categoria = ventas_completo[ventas_completo["Estado"] == "Completa"].groupby(["Fecha", "Categoría"])["Cantidad"].sum().reset_index()

# Llenar los días sin ventas con 0
df_completo = todas_combinaciones.merge(ventas_por_dia_categoria,
                                        how="left",
                                        on=["Fecha", "Categoría"]).fillna(0)

# Crear columna binaria: 1 si hubo ventas, 0 si no
df_completo['Hubo_Ventas'] = (df_completo['Cantidad'] > 0).astype(int)

# Pivotar para tener fechas como índice y categorías como columnas
heatmap_data = df_completo.pivot_table(
    index='Fecha', 
    columns='Categoría', 
    values='Hubo_Ventas',
    aggfunc='first'
).fillna(0)

# Ordenar las fechas
heatmap_data = heatmap_data.sort_index()

# Crear el heatmap binario
fig = px.imshow(
    heatmap_data.T, 
    labels={'x': 'Fecha', 'y': 'Categoría', 'color': 'Hubo Ventas'},
    x=heatmap_data.index.strftime("%Y-%m-%d"),
    y=heatmap_data.columns,
    color_continuous_scale=['lightcoral', 'lightgreen'],  # Rojo para 0, Verde para 1
    aspect="auto"
)

fig.update_layout(
    title="Heatmap Binario - Ventas por Categoría y Día (Verde = Hubo ventas, Rojo = No hubo)",
    xaxis_title="Fecha",
    yaxis_title="Categoría",
    width=1200,
    height=600
)

fig.update_xaxes(tickangle=45, tickmode='array', nticks=20)

fig.show()

In [54]:
df = ventas_completo[ventas_completo["Estado"] == "Completa"]

#Obtener todas las combinaciones posibles de fechas × categorías
fechas = df["Fecha"].unique()
categorias = df["Categoría"].unique()
todas_combinaciones = pd.DataFrame(list(itertools.product(fechas, categorias)),
                                   columns=["Fecha", "Categoría"])

#Sumar las cantidades reales vendidas por fecha y categoría
ventas_por_dia_categoria = df.groupby(["Fecha", "Categoría"])["Cantidad"].sum().reset_index()

# llenar los días sin ventas con 0
df_completo = todas_combinaciones.merge(ventas_por_dia_categoria,
                                        how="left",
                                        on=["Fecha", "Categoría"]).fillna(0)

tabla_dias_sin_compras = (
    df_completo.groupby("Categoría")["Cantidad"]
    .apply(lambda x: (x == 0).sum())
    .reset_index(name="Dias_sin_compras")
)

tabla_dias_sin_compras['percentage'] = (tabla_dias_sin_compras['Dias_sin_compras']/340)*100

tabla_dias_sin_compras

,Categoría,Dias_sin_compras,percentage
0,Bebidas,133,39.117647
1,Carnicería,115,33.823529
2,Congelados,140,41.176471
3,Conservas,153,45.000000
4,Frutas y Verduras,126,37.058824
5,Galletitas y Snacks,151,44.411765
6,Lácteos,127,37.352941
7,Panadería,132,38.823529


In [55]:
# GRÁFICO 1: ACUMULADO TOTAL

#agrupar por fecha y sumar CANTIDAD (unidades)
datos_unidades = ventas_completo.groupby('Fecha')[['Cantidad']].sum().reset_index()
datos_unidades = datos_unidades.sort_values('Fecha')

#calcular la suma acumulada de unidades
datos_unidades['Unidades_Acumuladas'] = datos_unidades['Cantidad'].cumsum()

fig_unidades = px.line(
    datos_unidades, 
    x='Fecha', 
    y='Unidades_Acumuladas',
    title='Crecimiento de Volumen de Ventas (Unidades Acumuladas)',
    labels={'Unidades_Acumuladas': 'Total Unidades Vendidas (u)', 'Fecha': 'Fecha'},
    markers=True
)

fig_unidades.update_layout(
    hovermode="x unified",
    xaxis_title="Tiempo",
    yaxis_title="Unidades (u)"
)

fig_unidades.show()

# GRÁFICO 2: DESGLOSE POR CATEGORÍA 
# agrupar por Fecha y Categoría usando CANTIDAD
datos_cat_unidades = ventas_completo.groupby(['Fecha', 'Categoría'])[['Cantidad']].sum().reset_index()
datos_cat_unidades = datos_cat_unidades.sort_values('Fecha')

# calcular acumulado por grupo
datos_cat_unidades['Unidades_Acumuladas'] = datos_cat_unidades.groupby('Categoría')['Cantidad'].cumsum()
fig_cat_unidades = px.line(
    datos_cat_unidades, 
    x='Fecha', 
    y='Unidades_Acumuladas', 
    color='Categoría',
    title='Unidades Acumuladas por Categoría de Producto',
    labels={'Unidades_Acumuladas': 'Unidades Acumuladas (u)'}
)

fig_cat_unidades.show()

# Modelo

## Regresión

In [56]:
def entrenar_regresion(df):
    df_reg = df.copy()
    
    df_reg['Dias'] = (df_reg['Fecha'] - df_reg['Fecha'].min()).dt.days
    
    X = df_reg[['Dias']]
    y = df_reg['Unidades_Acumuladas']
    
    # Train / Test split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.20, shuffle=False
    )
    
    # Modelo
    modelo = LinearRegression()
    modelo.fit(X_train, y_train)
    
    # Predicción completa
    df_reg['Predicción'] = modelo.predict(X)
    
    # Metricas
    mse = mean_squared_error(y_test, modelo.predict(X_test))
    rmse = np.sqrt(mse)   #Calculoo RMSE
    
    r2 = r2_score(y_test, modelo.predict(X_test))
    
    return df_reg, rmse, r2, modelo

In [57]:
df_total_pred, mse_total, r2_total, modelo_total = entrenar_regresion(datos_unidades)

print("RMSE total:", mse_total)
print("R2 total :", r2_total)

fig_reg = go.Figure()

fig_reg.add_trace(go.Scatter(
    x=df_total_pred['Fecha'], 
    y=df_total_pred['Unidades_Acumuladas'],
    mode='lines+markers',
    name='Ventas Acumuladas Reales'
))

fig_reg.add_trace(go.Scatter(
    x=df_total_pred['Fecha'], 
    y=df_total_pred['Predicción'],
    mode='lines',
    name='Regresión Lineal (Predicción)'
))

fig_reg.update_layout(
    title='Regresión Lineal sobre Ventas Acumuladas (Total)',
    xaxis_title='Fecha',
    yaxis_title='Unidades Acumuladas'
)

fig_reg.show()

RMSE total: 52.208781181232546
R2 total : 0.9920609011969701


In [58]:
metricas = []
df_predicciones = []

for cat, df_cat in datos_cat_unidades.groupby('Categoría'):
    
    df_pred, rmse, r2, modelo = entrenar_regresion(df_cat)  # ← rmse viene de la función
    df_pred['Categoría'] = cat
    
    df_predicciones.append(df_pred)
    metricas.append({'Categoría': cat, 'RMSE': rmse, 'R2': r2})  # ← guardar RMSE
      

df_reg_cat = pd.concat(df_predicciones, ignore_index=True)
df_metricas = pd.DataFrame(metricas)

print(df_metricas)

fig = px.line(
    df_reg_cat,
    x='Fecha',
    y='Unidades_Acumuladas',
    color='Categoría',
    title='Ventas Acumuladas Reales por Categoría'
)

fig_pred = px.line(
    df_reg_cat,
    x='Fecha',
    y='Predicción',
    color='Categoría',
    title='Predicción por Categoría (Regresión Lineal)',
    line_dash='Categoría'
)

fig.show()
fig_pred.show()

             Categoría       RMSE        R2
0              Bebidas  29.224829  0.837328
1           Carnicería  14.335241  0.974494
2           Congelados  26.648077  0.906910
3            Conservas  18.227941  0.914936
4    Frutas y Verduras  35.080863  0.802067
5  Galletitas y Snacks  32.052392  0.690701
6              Lácteos  46.623532  0.628150
7            Panadería  30.028081  0.864714


## Venta Proximos 20% de dias

In [79]:
# Gráficas
def plot_comparacion(df, titulo, marcar_corte=False):

    real_col = "Venta_Real" if "Venta_Real" in df.columns else "Cantidad"

    fig = make_subplots(rows=1, cols=1)

    fig.add_trace(go.Scatter(
        x=df["Fecha"], y=df[real_col],
        mode="lines", name="Real"
    ))

    fig.add_trace(go.Scatter(
        x=df["Fecha"], y=df["Prediccion"],
        mode="lines", name="Prediccion"
    ))

    if marcar_corte:
        f_corte = ventas_diarias_clean.iloc[corte]["Fecha"]
        fig.add_vline(x=f_corte)
        fig.add_annotation(x=f_corte, y=1, yref="paper",
                           text="Inicio Test", showarrow=False)

    fig.update_layout(
        title=titulo,
        xaxis_title="Fecha",
        yaxis_title="Unidades"
    )

    fig.show()


# Features
ventas_diarias = (
    ventas_completo
    .groupby("Fecha")["Cantidad"]
    .sum()
    .reset_index()
)

# Variables básicas
ventas_diarias["Dia_Semana"] = ventas_diarias["Fecha"].dt.dayofweek
ventas_diarias["Mes"] = ventas_diarias["Fecha"].dt.month

# Variables de dia específico 
ventas_diarias["EsLunes"] = (ventas_diarias["Dia_Semana"] == 0).astype(int)
ventas_diarias["EsMartes"] = (ventas_diarias["Dia_Semana"] == 1).astype(int)
ventas_diarias["EsMiercoles"] = (ventas_diarias["Dia_Semana"] == 2).astype(int)
ventas_diarias["EsJueves"] = (ventas_diarias["Dia_Semana"] == 3).astype(int)
ventas_diarias["EsViernes"] = (ventas_diarias["Dia_Semana"] == 4).astype(int)
ventas_diarias["EsSabado"] = (ventas_diarias["Dia_Semana"] == 5).astype(int)
ventas_diarias["EsDomingo"] = (ventas_diarias["Dia_Semana"] == 6).astype(int)

# Variables compuestas útiles
ventas_diarias["EsFinDeSemana"] = (ventas_diarias["Dia_Semana"] >= 5).astype(int)
ventas_diarias["EsInicioSemana"] = (ventas_diarias["Dia_Semana"] <= 1).astype(int)

# Variables de tendencia reciente
ventas_diarias["Lag1"] = ventas_diarias["Cantidad"].shift(1)  # Ayer
ventas_diarias["Lag2"] = ventas_diarias["Cantidad"].shift(2)  # Anteayer
ventas_diarias["Lag7"] = ventas_diarias["Cantidad"].shift(7)  # semana anterior

# Promedios móviles recientes
ventas_diarias["MA3"] = ventas_diarias["Cantidad"].rolling(3, min_periods=1).mean()  # Últimos 3 días
ventas_diarias["MA7"] = ventas_diarias["Cantidad"].rolling(7, min_periods=1).mean()  # Última semana
ventas_diarias["MA14"] = ventas_diarias["Cantidad"].rolling(14, min_periods=1).mean()  # Últimas 2 semanas

# Variables de estacionalidad mensual
ventas_diarias["EsInicioMes"] = (ventas_diarias["Fecha"].dt.day <= 7).astype(int)  # Primera semana
ventas_diarias["EsFinMes"] = (ventas_diarias["Fecha"].dt.day >= 25).astype(int)  # Última semana

# Features finales
features = [
    # Tendencias recientes
    "Lag1", "Lag2", "Lag7",
    "MA3", "MA7", "MA14",
    # Días específicos
   "EsMartes", "EsFinDeSemana","EsJueves","EsInicioSemana",
    # Patrones mensuales
    "EsFinMes"
]

# Limpiar solo las filas donde faltan lags (que son los primeros días)
ventas_diarias_clean = ventas_diarias.dropna(subset=["Lag1", "Lag2", "Lag7"]).reset_index(drop=True)

X = ventas_diarias_clean[features]
Y = ventas_diarias_clean["Cantidad"]

# División train/test 
corte = int(len(X) * 0.8)  # 80% entrenamiento, 20% test
X_train, X_test = X[:corte], X[corte:]
Y_train, Y_test = Y[:corte], Y[corte:]


# Entrenar y evaluar
def entrenar_y_evaluar(modelo, nombre_modelo):
    modelo.fit(X_train, Y_train)
    
    pred_train = modelo.predict(X_train)
    pred_test = modelo.predict(X_test)
    
    rmse_train = np.sqrt(mean_squared_error(Y_train, pred_train))
    rmse_test = np.sqrt(mean_squared_error(Y_test, pred_test))
    r2_train = r2_score(Y_train, pred_train)
    r2_test = r2_score(Y_test, pred_test)
    
    print(f"\n{nombre_modelo}")
    print(f"Train RMSE: {rmse_train:.2f}, R2: {r2_train:.4f}")
    print(f"Test RMSE: {rmse_test:.2f}, R2: {r2_test:.4f}")
    
    # DF con predicciones
    df_full = ventas_diarias_clean.copy()
    df_full["Prediccion"] = np.nan
    train_idx = X_train.index
    test_idx = X_test.index
    df_full.loc[train_idx, "Prediccion"] = pred_train
    df_full.loc[test_idx, "Prediccion"] = pred_test
    
    # DF de test
    df_test = ventas_diarias_clean.iloc[test_idx].copy()
    df_test["Venta_Real"] = Y_test
    df_test["Prediccion"] = pred_test
    
    return df_full, df_test

#Random Forest
modelo_rf = RandomForestRegressor(
    n_estimators=200,
    random_state=42,
    n_jobs=-1,
    max_depth=10
)

df_full_rf, df_test_rf = entrenar_y_evaluar(modelo_rf, "Random Forest")
plot_comparacion(df_full_rf, "Random Forest - Predicción Completa", marcar_corte=True)
plot_comparacion(df_test_rf, "Random Forest - Período de Test")

# Importancia de features
importancias = modelo_rf.feature_importances_
feature_importance_df = pd.DataFrame({
    'Feature': features,
    'Importance': importancias
}).sort_values('Importance', ascending=False)

print("Importancia de Features (en Random Forest):")
print(feature_importance_df)


Random Forest
Train RMSE: 2.24, R2: 0.9645
Test RMSE: 6.11, R2: 0.7011


Importancia de Features (en Random Forest):
           Feature  Importance
3              MA3    0.481820
1             Lag2    0.212521
0             Lag1    0.167335
4              MA7    0.043236
2             Lag7    0.040469
5             MA14    0.030830
7    EsFinDeSemana    0.005352
10        EsFinMes    0.005244
9   EsInicioSemana    0.004833
6         EsMartes    0.004494
8         EsJueves    0.003866


## Cancelación

In [129]:
Df_cancelaciones = ventas_completo.copy()
Df_cancelaciones = Df_cancelaciones[Df_cancelaciones["Estado"] != "Pendiente"]
mapeo = {"Completa": 0, "Cancelada": 1}
Df_cancelaciones["Cancelados"] = Df_cancelaciones["Estado"].map(mapeo)
mapeo_dias = {
    'Monday': 1,
    'Tuesday': 2,
    'Wednesday': 3,
    'Thursday': 4,
    'Friday': 5,
    'Saturday': 6,
    'Sunday': 7
}
Df_cancelaciones['Dia_Numerico'] = Df_cancelaciones['Dia_Semana'].map(mapeo_dias)
Df_cancelaciones = Df_cancelaciones.merge(clientes, on='ID_Cliente')
mapeo_regiones = {
    'Buenos Aires': 1,
    'Patagonia': 2,
    'Centro': 3,
    'Cuyo': 4,
    'NEA': 5,
    'NOA': 6
}
Df_cancelaciones['Region_Numerica'] = Df_cancelaciones['Región'].map(mapeo_regiones)

columnas_finales = [
    "ID_Cliente",
    "Cancelados", # Variable Objetivo
    "Método_Pago",
    "Venta_Total", 
    "Precio_Unitario",
    "Cantidad",
    "Dia_Numerico",
    "Mes",
    "Region_Numerica"
    ]

Df_modelado = Df_cancelaciones[columnas_finales]
Df_modelado

,ID_Cliente,Cancelados,Método_Pago,Venta_Total,Precio_Unitario,Cantidad,Dia_Numerico,Mes,Region_Numerica
0,10,0,1,77.25,15.45,5,3,1,1
1,106,0,4,5.65,5.65,1,3,1,5
2,235,0,3,46.35,15.45,3,3,1,5
3,114,0,1,17.55,3.51,5,3,1,3
4,132,0,4,26.05,5.21,5,3,1,1
...,...,...,...,...,...,...,...,...,...
2528,248,0,1,71.25,14.25,5,1,12,3
2529,44,0,4,48.72,8.12,6,1,12,1
2530,26,0,4,15.72,5.24,3,1,12,3
2531,246,0,3,33.69,11.23,3,1,12,5
